# Episode 1: Getting Started with AI Agents on Microsoft Foundry Portal

This notebook accompanies **Episode 1** of the AI-103 Hands-on series
("Defining an AI Agent on Microsoft Foundry Portal and the Basics of the Python SDK").

**Scenario:** An agent that uses the built-in **Web Search** tool to research
recent papers/articles on an LLM or AI Agent topic, list them with citations,
and then synthesize a cross-cutting summary of trends.

> Prerequisite: The agent `llm-paper-research-agent` has already been created
> in the Foundry Portal (Playground) with the Web Search tool enabled and the
> instructions shown below. This notebook calls that existing agent via the
> Python SDK (Responses API).

## 1. Setup

```bash
pip install azure-ai-projects azure-identity
az login
```

Set the following environment variable (or use a `.env` file with `python-dotenv`):

```
PROJECT_ENDPOINT=https://<your-resource>.services.ai.azure.com/api/projects/<your-project>
```

In [ ]:
import os
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

PROJECT_ENDPOINT = os.environ["PROJECT_ENDPOINT"]
AGENT_NAME = "llm-paper-research-agent"  # created earlier in Foundry Portal

project = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=DefaultAzureCredential(),
)
openai = project.get_openai_client()

print("Connected to project:", PROJECT_ENDPOINT)

## 2. Reference instructions (for reproducibility)

These are the exact instructions configured on the agent in the Portal.
They are included here so the agent can be recreated purely from code if needed
(see the optional "create from code" cell below).

In [ ]:
AGENT_INSTRUCTIONS = """You are a research assistant specialized in LLM (Large Language Model) and AI Agent topics.

When the user provides a topic, follow these two steps:

STEP 1 - Search and List:
Use web search to find recent, relevant papers or articles on the given topic.
For each result, present:
- Title
- Source (e.g., arXiv, conference name, publisher)
- A 1-2 sentence summary of the key contribution
- The source URL (always cite it)

Prioritize results from the last 12 months. Aim for 5-8 relevant results.

STEP 2 - Synthesize:
After listing all results, write a "Summary" section that identifies
3-5 common trends, themes, or notable points of discussion across the
papers you found. This should go beyond restating individual items —
highlight what they collectively suggest about the current state of
the field.

Always keep the two steps clearly separated and labeled in your response.
"""
print(AGENT_INSTRUCTIONS)

### (Optional) Create the agent from code instead of the Portal

Skip this cell if you already created the agent in the Portal.

In [ ]:
from azure.ai.projects.models import PromptAgentDefinition, WebSearchTool

# agent = project.agents.create_version(
#     agent_name=AGENT_NAME,
#     definition=PromptAgentDefinition(
#         model="gpt-4o",
#         instructions=AGENT_INSTRUCTIONS,
#         tools=[WebSearchTool()],
#     ),
#     description="Collects and synthesizes recent LLM/AI agent papers via web search.",
# )
# print(f"Agent created (id: {agent.id}, name: {agent.name}, version: {agent.version})")

## 3. Create a conversation

A `conversation` keeps the context across turns — this is the Responses API
equivalent of a "thread" in the older Assistants API.

In [ ]:
conversation = openai.conversations.create()
print(f"Conversation created (id: {conversation.id})")

## 4. Send a message and stream the response

`tool_choice="required"` isn't set here on purpose — we want the model to decide
on its own whether the query needs Web Search, based on the instructions.

In [ ]:
USER_INPUT = "Summarize the latest research on AI agent memory architectures."

stream_response = openai.responses.create(
    stream=True,
    conversation=conversation.id,
    input=USER_INPUT,
    extra_body={
        "agent_reference": {"name": AGENT_NAME, "type": "agent_reference"}
    },
)

full_text = ""
citations = []

for event in stream_response:
    if event.type == "response.output_text.delta":
        print(event.delta, end="", flush=True)
        full_text += event.delta
    elif event.type == "response.output_item.done":
        if event.item.type == "message":
            content = event.item.content[-1]
            if content.type == "output_text":
                for ann in content.annotations:
                    if ann.type == "url_citation":
                        citations.append(ann.url)
    elif event.type == "response.completed":
        print("\n\n--- Response completed ---")

## 5. Inspect the cited sources

In [ ]:
print("Cited URLs:")
for url in sorted(set(citations)):
    print(f"- {url}")

## 6. Where to see the tool-call trace

This Responses API architecture doesn't expose "Run Steps" the way the older
Assistants API (Threads/Runs) did. To inspect *when* the agent decided to call
Web Search and *what query* it generated, use the **Traces** tab in the Foundry
Portal for this agent, rather than polling run steps in code.

## Next steps

- Episode 2: Implementing basic RAG with the File Search tool
- Episode 3: Getting structured JSON output
- Episode 4: Responsible AI guardrails